# 10 — Thực nghiệm cuối: đóng băng cấu hình và chạy test một lần

- Trước khi đọc test: ghi mã băm của `configs/protocol.json`, mọi report validation và cấu hình mô hình đã chọn vào `reports/final/frozen.json`.
- Hai biến thể (D2): **train_only** (mô hình fit trên train, đúng như đã chọn) và **train_validation_refit** (biến thể chính: fit lại trên train + validation với đúng cấu hình; GRU4Rec/SR-GNN dùng số epoch đã chọn trên validation; vocabulary, popularity, láng giềng, luật và `known_at` tính lại trên train + validation). Reranker giữ nguyên mô hình học trên validation, chỉ thay ứng viên/đặc trưng bằng mô hình gốc của biến thể.
- Mô hình: Popularity, S-POP, Repeat-last, cấu hình tốt nhất của ItemKNN, SKNN, V-SKNN, STAN, VSTAN, SR, GRU4Rec, SR-GNN, reranker.
- Báo cáo: Hit/Recall/MRR/NDCG @10, @20; bài toán 1 và bài toán 2 (cart/mua ở nhóm kế tiếp); target mới/lặp; độ dài prefix; độ phổ biến của item target trong dữ liệu fit; thời gian fit/truy vấn; tỷ lệ fallback; khoảng tin cậy bootstrap 95% của NDCG@20; paired bootstrap so với Repeat-last và với mô hình kNN tốt nhất; biến thể phụ có luật `available`.
- **Diễn tập:** đặt biến môi trường `RR_FINAL_SPLIT=validation` để chạy toàn bộ đường code trên validation (refit chỉ 1 epoch, ghi vào `reports/final_rehearsal/`, không dùng để báo cáo). Lần chạy thật dùng mặc định `test` và chỉ chạy **một lần**.

## 1. Chế độ chạy, đóng băng cấu hình

In [ ]:
from pathlib import Path
import os, sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.samples import build_vocab, encode, prefix_events, UNK_IDX
from rr.models import popularity, prefix_matrix, repeat_last, spop
from rr.metrics import per_query_metrics, summarize, bootstrap_ci, paired_bootstrap
from rr.evaluation import load_samples, run, flat, fit_neighbour_model, recommend_neighbour
from rr import gru as G
from rr import srgnn as S
from rr.rerank import add_features, candidate_table, ranked_lists

SPLIT = os.environ.get("RR_FINAL_SPLIT", "test")
require(SPLIT in {"test", "validation"}, "RR_FINAL_SPLIT phải là test hoặc validation.")
REHEARSAL = SPLIT == "validation"
OUT = ROOT / ("reports/final_rehearsal" if REHEARSAL else "reports/final")
PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/eval"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
MAX_LEN = PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1
SEED = PROTOCOL["seed"]
BOOT = PROTOCOL["evaluation"]["bootstrap"]
WIDTH = 200
torch.set_num_threads(4)

if not REHEARSAL:
    require(not (OUT / "metrics.json").exists(), "Test đã được chạy; không chạy lại (protocol: test chạy đúng một lần).")
FROZEN_FILES = ["configs/protocol.json", "reports/samples/metrics.json"] + \
    [f"reports/eval/{n}_validation.json" for n in ["popularity", "spop", "repeat_last", "knn", "gru4rec", "srgnn", "reranker"]] + \
    ["models/gru4rec/config.json", "models/gru4rec/model.pt", "models/srgnn/config.json", "models/srgnn/model.pt",
     "models/reranker/config.json", "models/reranker/model.txt"]
frozen = {p: fingerprint(ROOT / p) for p in FROZEN_FILES}
OUT.mkdir(parents=True, exist_ok=True)
write_json(OUT / "frozen.json", {"generated_at_utc": utc_now(), "split": SPLIT, "files": frozen})
reports = {n: read_json(EVAL_DIR / f"{n}_validation.json") for n in ["popularity", "spop", "repeat_last", "knn", "gru4rec", "srgnn", "reranker"]}
for n, r in reports.items():
    require(r["protocol"]["version"] == PROTOCOL["version"], f"{n}_validation dùng protocol khác.")
print(f"Chế độ: {SPLIT}{' (DIỄN TẬP)' if REHEARSAL else ''} | đã đóng băng {len(frozen)} tệp vào {rel(OUT / 'frozen.json', ROOT)}")

## 2. Kiểm tra đầu vào và ví dụ biên

Ví dụ biên: bin độ phổ biến của target, paired bootstrap của hai danh sách giống nhau cho hiệu bằng 0.

In [ ]:
samples_report = read_json(ROOT / "reports/samples/metrics.json")
metadata_report = read_json(ROOT / "reports/metadata/metrics.json")
split_report = read_json(ROOT / "reports/session_split/metrics.json")
expected = {p: samples_report["artifacts"][p] for p in samples_report["artifacts"] if p.endswith(("_queries.parquet", "_targets.parquet"))}
for p in ["data/processed/item_property_history.parquet", "data/processed/item_catalog.parquet"]:
    expected[p] = metadata_report["artifacts"][p]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", len(inputs), "tệp")

POP_BINS = [(-1, 0, "unseen"), (1, 9, "1-9"), (10, 99, "10-99"), (100, None, "100+")]
def pop_bin(counts):
    counts = pd.Series(counts)
    out = pd.Series("", index=counts.index, dtype=object)
    for lo, hi, name in POP_BINS:
        out[counts.ge(lo) & (counts.le(hi) if hi is not None else True)] = name
    return out.to_numpy()
assert pop_bin([0, 1, 9, 10, 150]).tolist() == ["unseen", "1-9", "1-9", "10-99", "100+"]
assert paired_bootstrap([0.5, 0.2], [0.5, 0.2], 50, 0)["mean_diff"] == 0
print("Đã kiểm tra ví dụ biên.")

## 3. Dữ liệu chung

In [ ]:
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
eval_q, eval_t = load_samples(ROOT, SPLIT)
eval_events = events[events["split"] == SPLIT]
qids, qtimes = eval_q["query_id"].to_numpy(), eval_q["query_time_ms"].to_numpy()
prefix_full = prefix_events(eval_q, eval_events, ("itemid", "event"))
prefix = prefix_full[prefix_full["distance"] < MAX_GROUPS]
seen = prefix_matrix(prefix_full, qids)
knn_sel = reports["knn"]["selected"]
FAMILIES = ["itemknn", "sknn", "vsknn", "stan", "vstan", "sr"]
best_knn = max(FAMILIES, key=lambda f: reports["knn"]["runs"][knn_sel[f]["best_run"]]["all"]["ndcg@20"])
print(f"{SPLIT}: {len(eval_q):,} query | kNN tốt nhất trên validation: {best_knn}")

VARIANTS = {"train_only": {"fit_splits": ["train"], "known_at": "known_at_train_ms"},
            "train_validation_refit": {"fit_splits": ["train", "validation"], "known_at": "known_at_trainval_ms"}}
if REHEARSAL:
    VARIANTS["train_validation_refit"]["rehearsal_epochs"] = 1

## 4. Chạy từng biến thể

Với mỗi biến thể: fit mọi mô hình trên dữ liệu fit, sinh danh sách ứng viên cho query đánh giá, rồi qua cùng bộ lọc và fallback.

In [ ]:
def target_pop_subsets(targets, fit_events):
    counts = fit_events.groupby("itemid").size()
    t = targets.copy()
    t["pop_bin"] = pop_bin(counts.reindex(t["itemid"].to_numpy()).fillna(0).to_numpy())
    return t

all_results, per_query_ndcg, timings, saved_topk = {}, {}, {}, []
for vname, vcfg in VARIANTS.items():
    t_var = time.perf_counter()
    fit_events = events[events["split"].isin(vcfg["fit_splits"])].reset_index(drop=True)
    require(fit_events["timestamp"].max() < qtimes.min(), "Dữ liệu fit phải nằm trước mọi query đánh giá.")
    vocab = build_vocab(fit_events)
    flt = CandidateFilter.from_frames(vocab, catalog, history, vcfg["known_at"])
    flt_av = CandidateFilter.from_frames(vocab, catalog, history, vcfg["known_at"], use_availability=True)
    sel = reports["popularity"]["selected"]
    fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(fit_events, sel["fallback_weights"]))
    pop_items, pop_scores = (a[:FALLBACK_TOP_N] for a in popularity(fit_events, sel["popularity_behavior_weights"]))
    targets_pb = target_pop_subsets(eval_t, fit_events)
    lists, fit_s, query_s = {}, {}, {}

    def timed(name, fit_fn, rec_fn):
        t0 = time.perf_counter(); model = fit_fn(); fit_s[name] = time.perf_counter() - t0
        t0 = time.perf_counter(); lists[name] = rec_fn(model); query_s[name] = time.perf_counter() - t0
        return model

    EMPTY = np.empty((len(eval_q), 0), dtype=np.int64)
    lists["popularity"], fit_s["popularity"], query_s["popularity"] = (EMPTY, None), 0.0, 0.0
    best_spop = reports["spop"]["runs"][reports["spop"]["best"]]["weights"]
    timed("spop", lambda: None, lambda m: spop(prefix_full, qids, best_spop, fb_items, fb_scores, width=50))
    timed("repeat_last", lambda: None, lambda m: repeat_last(prefix_full, qids, width=50))
    knn_models = {}
    for fam in FAMILIES:
        knn_models[fam] = timed(fam, lambda fam=fam: fit_neighbour_model(fam, knn_sel, fit_events),
                                lambda m, fam=fam: recommend_neighbour(m, fam, knn_sel, prefix, qids, qtimes, WIDTH))

    # GRU4Rec
    gcfg = read_json(ROOT / "models/gru4rec/config.json")
    def fit_gru():
        if vname == "train_only":
            m = G.build_from_config(gcfg); m.load_state_dict(torch.load(ROOT / "models/gru4rec/model.pt")); return m
        data = G.GroupData(fit_events, vocab)
        torch.manual_seed(SEED)
        m = G.build_from_config(gcfg, n_items=int(vocab["item_idx"].max()) + 1)
        G.fit(m, data, data.windows(MAX_LEN), vcfg.get("rehearsal_epochs", gcfg["epochs"]), lr=gcfg["lr"],
              batch_size=gcfg["batch_size"], n_neg=gcfg["n_neg"], seed=SEED, loss=gcfg["loss"], bpreg=gcfg["bpreg"],
              elu=gcfg["elu"], sample_alpha=gcfg["sample_alpha"])
        return m
    gru = timed("gru4rec", fit_gru, lambda m: G.topn_items(m, G.query_states(m, G.GroupData(eval_events, vocab), eval_q),
                                                            G.idx_to_itemid(vocab), WIDTH))

    # SR-GNN
    scfg = read_json(ROOT / "models/srgnn/config.json")
    def fit_srgnn():
        n_items = int(vocab["item_idx"].max()) + 1
        m = S.SRGNN(n_items, dim=scfg["dim"], steps=scfg["steps"], dropout=scfg["dropout"])
        if vname == "train_only":
            m.load_state_dict(torch.load(ROOT / "models/srgnn/model.pt")); return m
        fq, ft = [], []
        for s in vcfg["fit_splits"]:
            q_, t_ = load_samples(ROOT, s); fq.append(q_); ft.append(t_)
        fq, ft = pd.concat(fq, ignore_index=True), pd.concat(ft, ignore_index=True)
        graphs = S.GraphData(prefix_events(fq, fit_events, ("itemid", "event"), max_groups=MAX_GROUPS), fq["query_id"].to_numpy(), vocab)
        ptr, val = S.target_csr(ft, fq["query_id"].to_numpy(), vocab)
        keep = np.flatnonzero(np.diff(ptr) > 0)  # every fit query has an in-vocabulary target here
        require(len(keep) == len(fq), "Có query fit không có target trong vocabulary fit.")
        neg, lq = S.negative_distribution(val, n_items, scfg["sample_alpha"])
        S.fit(m, graphs, ptr, val, neg, lq, vcfg.get("rehearsal_epochs", scfg["epochs"]), lr=scfg["lr"],
              batch_size=scfg["batch_size"], n_neg=scfg["n_neg"], seed=SEED)
        return m
    timed("srgnn", fit_srgnn, lambda m: G.topn_items(m, S.session_vectors(m, S.GraphData(prefix, qids, vocab)),
                                                      G.idx_to_itemid(vocab), WIDTH))

    # Reranker (model trained on validation in notebook 09; candidates from this variant's base models)
    rcfg = read_json(ROOT / "models/reranker/config.json")
    booster = lgb.Booster(model_str=(ROOT / "models/reranker/model.txt").read_text(encoding="utf-8"))
    def rerank_lists(_):
        src = {"knn": lists[rcfg["sources"]["knn"]], "gru": lists["gru4rec"]}
        table = add_features(candidate_table(qids, src, prefix, top_n=rcfg["top_n"]), eval_q, prefix, fit_events, history)
        return ranked_lists(table, booster.predict(table[rcfg["features"]]), qids, WIDTH)
    timed("reranker", lambda: None, rerank_lists)

    # Evaluate every model once
    res = {}
    for name, (items, scores) in lists.items():
        if name == "popularity":
            rep, pq, topk, _ = run(EMPTY, None, eval_q, eval_t, flt, pop_items, pop_scores)
            rep_av, *_ = run(EMPTY, None, eval_q, eval_t, flt_av, pop_items, pop_scores)
            rep_ex, *_ = run(EMPTY, None, eval_q, eval_t, flt, pop_items, pop_scores, exclude=seen)
        else:
            rep, pq, topk, isfb = run(items, scores, eval_q, eval_t, flt, fb_items, fb_scores)
            if vname == "train_only" and name in {best_knn, "gru4rec"}:
                rows, cols = np.nonzero(topk != -1)
                saved_topk.append(pd.DataFrame({"model": name, "query_id": qids[rows], "rank": cols + 1,
                                                "itemid": topk[rows, cols], "is_fallback": isfb[rows, cols]}))
            rep_av, *_ = run(items, scores, eval_q, eval_t, flt_av, fb_items, fb_scores)
            rep_ex = None if name == "repeat_last" else run(items, scores, eval_q, eval_t, flt, fb_items, fb_scores, exclude=seen)[0]
        rep["by_target_popularity"] = {}
        for _, _, b in POP_BINS:
            sub = targets_pb[targets_pb["pop_bin"] == b]
            keep = np.isin(qids, sub["query_id"].unique())
            if keep.any():
                rep["by_target_popularity"][b] = summarize(per_query_metrics(topk[keep], qids[keep], sub))
        rep["bootstrap_ndcg@20"] = bootstrap_ci(pq["ndcg@20"].to_numpy(), BOOT["n_resamples"], BOOT["seed"], BOOT["level"])
        rep["fit_seconds"] = round(fit_s[name] or 0.0, 1)
        rep["query_ms_per_query"] = round(1000 * query_s[name] / len(eval_q), 4)
        res[name] = {"main": rep, "with_availability": rep_av, "exclude_seen": rep_ex}
        per_query_ndcg[(vname, name)] = pq["ndcg@20"].to_numpy()
    for name in res:
        res[name]["main"]["paired_vs_repeat_last"] = paired_bootstrap(per_query_ndcg[(vname, name)], per_query_ndcg[(vname, "repeat_last")],
                                                                        BOOT["n_resamples"], BOOT["seed"], BOOT["level"])
        res[name]["main"][f"paired_vs_{best_knn}"] = paired_bootstrap(per_query_ndcg[(vname, name)], per_query_ndcg[(vname, best_knn)],
                                                                       BOOT["n_resamples"], BOOT["seed"], BOOT["level"])
    all_results[vname] = res
    timings[vname] = round(time.perf_counter() - t_var, 1)
    print(f"{vname}: xong trong {timings[vname] / 60:.1f} phút")
    display(flat({n: r["main"] for n, r in res.items()}).style.format("{:.4f}"))

## 5. Bảng kết quả

In [ ]:
MAIN = "train_validation_refit"
main = {n: r["main"] for n, r in all_results[MAIN].items()}
for part, title in [("all", "Bài toán 1 — tương tác kế tiếp"), ("strong_behavior_targets", "Bài toán 2 — thêm giỏ/mua kế tiếp"),
                    ("new_targets", "Target mới"), ("repeat_targets", "Target lặp")]:
    print(title)
    display(flat(main, part).style.format("{:.4f}"))
display(pd.DataFrame({n: {"ndcg@20": r["bootstrap_ndcg@20"]["mean"], "ci_low": r["bootstrap_ndcg@20"]["low"],
                          "ci_high": r["bootstrap_ndcg@20"]["high"], "diff_vs_repeat_last": r["paired_vs_repeat_last"]["mean_diff"],
                          "diff_low": r["paired_vs_repeat_last"]["low"], "diff_high": r["paired_vs_repeat_last"]["high"],
                          "fallback_slots": r["fallback"]["slot_rate"], "fit_s": r["fit_seconds"], "ms/query": r["query_ms_per_query"]}
                      for n, r in main.items()}).T.style.format("{:.4f}"))

## 6. Lưu báo cáo và bảng Markdown

In [ ]:
def md_table(rows, cols, fmt="{:.4f}"):
    head = "| Mô hình | " + " | ".join(cols) + " |\n|" + " --- |" * (len(cols) + 1) + "\n"
    return head + "".join(f"| {n} | " + " | ".join(fmt.format(v.get(c, float('nan'))) for c in cols) + " |\n" for n, v in rows.items())

cols = ["hit@10", "recall@10", "mrr@10", "ndcg@10", "hit@20", "recall@20", "mrr@20", "ndcg@20"]
lines = [f"# Kết quả thực nghiệm cuối trên {SPLIT}{' (DIỄN TẬP — không dùng để báo cáo)' if REHEARSAL else ''}\n",
         f"Sinh bởi `notebooks/10_thuc_nghiem_cuoi_test.ipynb` lúc {utc_now()}, protocol {PROTOCOL['version']}. "
         f"{len(eval_q):,} query; mẫu số gồm cả query có target ngoài vocabulary. Biến thể chính: `{MAIN}`.\n"]
for vname in VARIANTS:
    res = {n: r["main"] for n, r in all_results[vname].items()}
    lines.append(f"\n## Biến thể `{vname}`\n")
    for part, title in [("all", "Bài toán 1 — tương tác kế tiếp"), ("strong_behavior_targets", "Bài toán 2 — thêm giỏ/mua ở nhóm kế tiếp"),
                        ("new_targets", "Chỉ target mới"), ("repeat_targets", "Chỉ target lặp")]:
        lines.append(f"\n### {title}\n\n" + md_table({n: r.get(part, {}) for n, r in res.items()}, cols))
    lines.append("\n### NDCG@20: khoảng tin cậy 95% và paired bootstrap so với Repeat-last\n\n" + md_table(
        {n: {"NDCG@20": r["bootstrap_ndcg@20"]["mean"], "CI thấp": r["bootstrap_ndcg@20"]["low"], "CI cao": r["bootstrap_ndcg@20"]["high"],
             "Δ vs Repeat-last": r["paired_vs_repeat_last"]["mean_diff"], "Δ thấp": r["paired_vs_repeat_last"]["low"],
             "Δ cao": r["paired_vs_repeat_last"]["high"]} for n, r in res.items()},
        ["NDCG@20", "CI thấp", "CI cao", "Δ vs Repeat-last", "Δ thấp", "Δ cao"]))
    lines.append("\n### NDCG@20 theo độ dài prefix (nhóm)\n\n" + md_table({n: {b: v["ndcg@20"] for b, v in r["by_prefix_len"].items()} for n, r in res.items()},
                                                                       ["1", "2-4", "5-9", "10+"]))
    lines.append("\n### NDCG@20 theo độ phổ biến của item target trong dữ liệu fit\n\n" + md_table(
        {n: {b: v["ndcg@20"] for b, v in r["by_target_popularity"].items()} for n, r in res.items()}, [b for _, _, b in POP_BINS]))
    lines.append("\n### Thời gian và fallback\n\n" + md_table(
        {n: {"fit (s)": r["fit_seconds"], "ms/query": r["query_ms_per_query"], "tỷ lệ ô fallback": r["fallback"]["slot_rate"]} for n, r in res.items()},
        ["fit (s)", "ms/query", "tỷ lệ ô fallback"], "{:.3f}"))
    lines.append("\n### Biến thể phụ có luật `available ≠ 0` (NDCG@20)\n\n" + md_table(
        {n: {"chính": r["main"]["all"]["ndcg@20"], "có luật available": r["with_availability"]["all"]["ndcg@20"]} for n, r in all_results[vname].items()},
        ["chính", "có luật available"]))
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
require(all(fingerprint(ROOT / p) == v for p, v in frozen.items()), "Cấu hình đóng băng bị thay đổi trong lúc chạy.")
from rr.io import save_verified
topk_info = save_verified(pd.concat(saved_topk, ignore_index=True), PROCESSED / f"final/{SPLIT}_train_only_topk.parquet",
                          ["model", "query_id", "rank"], ROOT)
write_json(OUT / "metrics.json", {"saved_topk": topk_info, "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
                                  "lightgbm_version": lgb.__version__, "notebook": "notebooks/10_thuc_nghiem_cuoi_test.ipynb",
                                  "split": SPLIT, "rehearsal": REHEARSAL, "protocol": {"version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
                                  "frozen": frozen, "inputs": inputs, "n_queries": len(eval_q), "best_knn_on_validation": best_knn,
                                  "primary_variant": MAIN, "variant_seconds": timings, "results": all_results, "status": "PASS"})
(OUT / "BAO_CAO_KET_QUA.md").write_text("".join(lines), encoding="utf-8", newline="\n")
print("Báo cáo:", rel(OUT / "metrics.json", ROOT), rel(OUT / "BAO_CAO_KET_QUA.md", ROOT))